# 2. Validation against EMTP-RV: no coupling impedance needed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WillemLam/unified-acdc-powerflow/blob/main/benchmarks/pandapower_acdc/02_emtp_rv_microgrid.ipynb)

pandapower's AC/DC power flow puts every converter behind a coupling impedance; a direct connection is
approximated with a small one (a dummy bus with a tiny impedance). Is an impedance needed at all to model a
converter correctly? We check against an independent reference: an
**EMTP-RV** time-domain simulation of the 26-bus hybrid AC/DC microgrid of the EPFL laboratory, including the
converters and their control loops ([EMTP-RV model](https://github.com/DESL-EPFL/Hybrid_ACDC_EMTP_simulation)).
This is the validation of W. Lambrichts and M. Paolone, "General and Unified Model of the Power Flow Problem in Multiterminal AC/DC Networks", IEEE Trans. Power Systems, 2024, [doi:10.1109/TPWRS.2024.3378926](https://doi.org/10.1109/TPWRS.2024.3378926), section IV-A.

Both power flows get the same grid and the injections measured in EMTP-RV, and their voltages are compared with
the EMTP-RV voltages.

![Grid topology](https://raw.githubusercontent.com/WillemLam/unified-acdc-powerflow/main/validation/figures/microgrid.png)

In [1]:
# Setup. In Colab: download the repository and install pandapower 3.5.5.
# Locally: use the repository this notebook is in.
import os, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("unified-acdc-powerflow").exists():
        !git clone -q https://github.com/WillemLam/unified-acdc-powerflow.git
    %cd unified-acdc-powerflow
    !pip install -q "pandapower==3.5.5" pyyaml
else:
    root = Path.cwd()
    while not (root / "pyproject.toml").exists():
        root = root.parent
    os.chdir(root)
sys.path.insert(0, os.getcwd())

import logging, time, warnings
logging.getLogger("pandapower").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import pandapower as pp
from unified_acdc_powerflow import run_pf
from unified_acdc_powerflow.cases import load_case

def run_pandapower(net, init="auto"):
    """pandapower's own AC/DC power flow: a status line with the iterations and time reported by pandapower."""
    t0 = time.perf_counter()
    try:
        pp.runpp(net, init=init, max_iteration=100, tolerance_mva=1e-8)
    except Exception as err:
        return f"{type(err).__name__}: {str(err)[:70]} | time: {(time.perf_counter() - t0) * 1e3:.1f} ms"
    return f"converged | iterations: {net._ppc['iterations']} | time: {net._ppc['et'] * 1e3:.1f} ms"

def report(res):
    """The same status line for unified-acdc-powerflow."""
    return (f"{res.message} | iterations: {res.iterations} | max mismatch: {res.max_mismatch:.2e} | "
            f"time: {res.solve_time_s * 1e3:.1f} ms")

def with_coupling_impedance(net, z):
    """A copy of net for pandapower: every converter behind a coupling impedance of z p.u. (AC: r = x; DC: r)."""
    n = pp.from_json_string(pp.to_json(net))
    for i, v in n.vsc.iterrows():
        z_ac = z * n.bus.vn_kv[v.bus] ** 2 / n.sn_mva
        z_dc = z * n.bus_dc.vn_kv[v.bus_dc] ** 2 / n.sn_mva
        n.vsc.loc[i, ["r_ohm", "x_ohm", "r_dc_ohm"]] = [z_ac, z_ac, z_dc]
    return n

print("pandapower", pp.__version__)


pandapower 3.5.5


## Step 1: the EMTP-RV reference

The balanced case of the paper. For each bus, the voltage and the injected power at the end of the EMTP-RV
simulation, in p.u. (100 kVA; 400 V AC, 800 V DC). The case is balanced, so phase a represents the AC side.

In [2]:
emtp = pd.read_csv("validation/emtp/emtp_balanced.csv")
E_emtp = dict(zip(emtp.bus, emtp.E_re_pu + 1j * emtp.E_im_pu))
emtp

,bus,side,E_re_pu,E_im_pu,P_pu,Q_pu
0,B01,AC,1.000000,-1.183948e-09,9.405075e-01,-3.439228e-01
1,B02,AC,0.990604,-9.004772e-03,0.000000e+00,0.000000e+00
2,B03,AC,0.971491,-9.627963e-03,-3.000001e-01,-9.425143e-05
3,B04,AC,0.987752,-1.271954e-02,0.000000e+00,0.000000e+00
4,B05,AC,0.981991,-1.242763e-02,-3.749766e-01,-7.511781e-02
5,B06,AC,0.985940,-2.100600e-02,0.000000e+00,0.000000e+00
6,B07,AC,0.985422,-2.337339e-02,0.000000e+00,0.000000e+00
7,B08,AC,0.984246,-2.182580e-02,0.000000e+00,0.000000e+00
8,B09,AC,0.984550,-2.231191e-02,-1.551741e-07,-1.718092e-15
9,B10,AC,0.967771,-4.176449e-02,0.000000e+00,0.000000e+00


## Step 2: the grid, with the EMTP-RV setpoints

The case file `validation/cases/microgrid_emtp.yaml` holds the line data and the EMTP-RV injections as setpoints:
- the slack voltage at B01;
- the loads at B03, B05, B11 and B13;
- the DC injections at B23–B26: two sources and two loads of 5 kW;
- two converters in PQ mode;
- two converters holding the DC voltage (VdcQ).

The converters are connected directly to their buses.

In [3]:
net = load_case("microgrid_emtp")
display(net.load_dc)
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu"]]

,name,bus_dc,p_dc_mw,scaling,in_service,controllable,type
0,None,23,-0.005,1.0,True,False,None
1,None,24,-0.005,1.0,True,False,None
2,None,25,0.005,1.0,True,False,None
3,None,26,0.005,1.0,True,False,None


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu
0,IC 21-17,17,21,PQ,-0.001717,0.01,1.000000
1,IC 19-15,15,19,PQ,0.001708,0.01,1.000000
2,IC 22-18,18,22,VdcQ,0.000000,0.01,0.999993
3,IC 20-16,16,20,VdcQ,0.000000,0.01,0.999993


In [4]:
def error_vs_emtp(E_ac, E_dc):
    """Largest |E_EMTP-RV - E| per grid, from dictionaries bus number -> complex (AC) / real (DC) voltage."""
    rows = [(b, "AC", abs(E_emtp[b] - E_ac[int(b[1:])])) for b in emtp.bus[emtp.side == "AC"]]
    rows += [(b, "DC", abs(E_emtp[b] - E_dc[int(b[1:])])) for b in emtp.bus[emtp.side == "DC"]]
    err = pd.DataFrame(rows, columns=["bus", "side", "|ΔE| (p.u.)"])
    return {side: err[err.side == side].iloc[:, 2].max() for side in ("AC", "DC")}


## Step 3: unified-acdc-powerflow, converters connected directly

For comparison, the paper (Table III) reports a largest voltage error of 7.4e-6 p.u. (AC) and 5.9e-8 p.u. (DC).

In [5]:
res = run_pf(net, tol=1e-10)
m = res.model
ours = error_vs_emtp({b: res.E_ac[m.bus_lookup[b]] for b in net.bus.index},
                     {b: res.E_dc[m.bus_dc_lookup[b]] for b in net.bus_dc.index})
print("unified-acdc-powerflow:", report(res))
print(f"largest |ΔE| vs EMTP-RV: AC {ours['AC']:.1e} p.u., DC {ours['DC']:.1e} p.u.")

unified-acdc-powerflow: converged | iterations: 5 | max mismatch: 7.03e-14 | time: 52.6 ms
largest |ΔE| vs EMTP-RV: AC 3.3e-06 p.u., DC 3.2e-08 p.u.


Without any coupling impedance, unified-acdc-powerflow reproduces the EMTP-RV steady state to a few 1e-6 p.u.

## Step 4: pandapower

pandapower needs a non-zero coupling impedance: with zero it stops with a division by zero, and with a very small
one (1e-6 p.u.) it does not converge. With 1e-3 p.u. it works.

In [6]:
def pandapower_voltages(n):
    E_ac = {b: n.res_bus.vm_pu[b] * np.exp(1j * np.radians(n.res_bus.va_degree[b])) for b in n.bus.index}
    return E_ac, dict(n.res_bus_dc.vm_pu)

for z in (0.0, 1e-6):
    print(f"pandapower, coupling impedance {z:g} p.u.:", run_pandapower(with_coupling_impedance(net, z)))
pp_net = with_coupling_impedance(net, 1e-3)
print("pandapower, coupling impedance 1e-3 p.u.:", run_pandapower(pp_net))
theirs = error_vs_emtp(*pandapower_voltages(pp_net))
print(f"largest |ΔE| vs EMTP-RV: AC {theirs['AC']:.1e} p.u., DC {theirs['DC']:.1e} p.u.")

pandapower, coupling impedance 0 p.u.: FloatingPointError: divide by zero encountered in divide | time: 34.3 ms


pandapower, coupling impedance 1e-06 p.u.: LoadflowNotConverged: Power Flow nr did not converge after 100 iterations! | time: 562.9 ms


pandapower, coupling impedance 1e-3 p.u.: converged | iterations: 4 | time: 29.3 ms
largest |ΔE| vs EMTP-RV: AC 4.0e-03 p.u., DC 1.9e-03 p.u.


With 1e-3 p.u. pandapower converges, but its voltages are about 1000 times further from EMTP-RV than those
of unified-acdc-powerflow. The DC powers show why. Per DC bus, the power injected into the DC network (converter
DC side at B19–B22, DC sources and loads at B23–B26):

In [7]:
dc = emtp[emtp.side == "DC"]
p_pp = {b: 0.0 for b in pp_net.bus_dc.index}
for i, l in pp_net.line_dc.iterrows():            # from pandapower's own line results
    p_pp[l.from_bus_dc] += pp_net.res_line_dc.p_from_mw[i]
    p_pp[l.to_bus_dc] += pp_net.res_line_dc.p_to_mw[i]
pd.DataFrame({"bus": dc.bus.values,
              "EMTP-RV (kW)": dc.P_pu.values * 100.0,
              "unified-acdc-powerflow (kW)": [res.P_dc[m.bus_dc_lookup[int(b[1:])]] * 100.0 for b in dc.bus],
              "pandapower (kW)": [p_pp[int(b[1:])] * 1e3 for b in dc.bus]}).round(2)

,bus,EMTP-RV (kW),unified-acdc-powerflow (kW),pandapower (kW)
0,B19,-1.71,-1.71,-1.71
1,B20,-3.07,-3.07,-0.72
2,B21,1.72,1.72,1.72
3,B22,3.09,3.09,0.71
4,B23,5.00,5.00,0.50
5,B24,5.00,5.00,0.50
6,B25,-5.00,-5.00,-0.50
7,B26,-5.00,-5.00,-0.50


In pandapower's solution the DC sources and loads at B23–B26 inject ±0.5 kW instead of ±5 kW: it multiplies DC
loads by the MVA base (here 0.1 MVA).

## Step 5: with the DC loads corrected, the coupling impedance remains

We cancel the DC-load scaling and vary pandapower's coupling impedance.

In [8]:
rows = [("unified-acdc-powerflow (direct connection)", ours["AC"], ours["DC"])]
for z in (1e-4, 1e-3, 1e-2):
    n = with_coupling_impedance(net, z)
    n.load_dc["p_dc_mw"] = n.load_dc.p_dc_mw / n.sn_mva
    status = run_pandapower(n)
    e = error_vs_emtp(*pandapower_voltages(n)) if status.startswith("converged") else {"AC": np.nan, "DC": np.nan}
    rows.append((f"pandapower, DC loads corrected, coupling impedance {z:g} p.u.", e["AC"], e["DC"]))
pd.DataFrame(rows, columns=["power flow", "largest |ΔE| AC (p.u.)", "largest |ΔE| DC (p.u.)"])

,power flow,largest |ΔE| AC (p.u.),largest |ΔE| DC (p.u.)
0,unified-acdc-powerflow (direct connection),0.000003,3.248613e-08
1,"pandapower, DC loads corrected, coupling imped...",0.000007,4.389809e-06
2,"pandapower, DC loads corrected, coupling imped...",0.000011,4.389809e-06
3,"pandapower, DC loads corrected, coupling imped...",0.000054,4.389809e-06


pandapower's AC error grows with its coupling impedance: the impedance is a modelling artefact, not part of the
physics. The closest agreement with EMTP-RV comes from connecting the converters directly. (pandapower's DC error
stays at about 4e-6 p.u. for another reason: it gets no converter losses here, while unified-acdc-powerflow uses
the small losses measured in EMTP-RV, as in the paper.)

**Takeaway.**
- A converter does not need a coupling impedance in the power flow. unified-acdc-powerflow connects converters
  directly and reproduces an EMTP-RV simulation to a few 1e-6 p.u.
- pandapower 3.5.5 approximates a direct connection with a small coupling impedance, so its result depends on
  that choice. On this grid it converges, but its DC powers are wrong by a factor of 10 because it scales DC
  loads by the MVA base.